# 문제해결 1: 금융사기 보고서 기반 FAISS Naive RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: 노트북과 같은 폴더의 `진화하는 금융사기, 모두의 경계가 필요한 시점.pdf` (또는 `data/` 폴더)

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | PDF 페이지를 `Document` 객체로 정상 로딩한다. |
| 청킹 | 문서가 적절한 크기의 청크로 분할된다. |
| 벡터DB 생성 | FAISS 벡터스토어가 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [3]:
# uv add langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv pandas

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `PyPDFLoader`: PDF 문서 로딩
- `RecursiveCharacterTextSplitter`: 문서 청킹
- `OpenAIEmbeddings`: 텍스트 임베딩
- `FAISS`: 로컬 벡터스토어
- `ChatOpenAI`: 답변 생성 LLM

In [4]:
from pathlib import Path
import os
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

/var/folders/95/t3btf0056ng73bjnh02tb0tr0000gn/T/ipykernel_21240/1765998713.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


OpenAI API를 사용할 때는 `.env`에 `OPENAI_API_KEY`를 설정한다. 프로젝트의 OpenAI 호환 서버를 사용할 때는 `LLM_API_KEY`, `LLM_BASE_URL`을 설정한다. 서버는 채팅과 임베딩 API를 모두 지원해야 한다. 필요하면 `CHAT_MODEL`, `EMBEDDING_MODEL`로 모델명을 지정한다.

In [5]:
load_dotenv(dotenv_path=Path('.env'), override=True)

# 프로젝트의 OpenAI 호환 서버 설정을 우선한다.
MONO_API_KEY = os.getenv('LLM_API_KEY')
MONO_BASE_URL = os.getenv('LLM_BASE_URL')
if MONO_API_KEY:
    if not MONO_BASE_URL:
        raise ValueError('LLM_API_KEY가 설정되어 있으면 LLM_BASE_URL도 필요합니다.')
    API_KEY = MONO_API_KEY
    BASE_URL = MONO_BASE_URL
else:
    API_KEY = os.getenv('OPENAI_API_KEY')
    BASE_URL = None

if not API_KEY:
    raise ValueError('LLM_API_KEY 또는 OPENAI_API_KEY를 .env/환경변수에 설정하세요.')

print('API 설정 확인 완료:', BASE_URL or 'OpenAI 기본 주소')

API 설정 확인 완료: https://monogpt.kr/api/monorouter/v1


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 둔다. macOS에서 파일명이 유니코드 분해형으로 저장된 경우도 찾을 수 있도록 정규화해서 비교한다.

In [6]:
import unicodedata

PDF_NAME = '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf'
PDF_PATH = next(
    (
        path
        for folder in (Path('.'), Path('data'))
        if folder.is_dir()
        for path in folder.iterdir()
        if path.is_file() and unicodedata.normalize('NFC', path.name) == PDF_NAME
    ),
    None,
)
if PDF_PATH is None:
    raise FileNotFoundError(f'{PDF_NAME} 파일을 노트북 폴더 또는 data 폴더에서 찾을 수 없습니다.')

print('PDF_PATH:', PDF_PATH)

PDF_PATH: 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

PDF를 페이지 단위 `Document`로 로딩한다.  
각 페이지는 `page_content`와 `metadata`를 가진다.

### pdf 파일 로딩

In [7]:
# PDF 페이지를 Document 객체로 로드
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()
if not pages:
    raise ValueError(f'PDF에서 페이지를 읽지 못했습니다: {PDF_PATH}')

for doc in pages:
    doc.metadata['source_name'] = PDF_NAME
    doc.metadata['page_label'] = doc.metadata.get('page', 0) + 1

print('로드된 페이지 수:', len(pages))
print('첫 번째 페이지 메타데이터:')
pprint(pages[0].metadata)
print('\n첫 번째 페이지 내용 일부:')
print(pages[0].page_content[:500])

로드된 페이지 수: 15
첫 번째 페이지 메타데이터:
{'author': 'user',
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 페이지 내용 일부:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 5. 문서 청킹

RAG에서는 문서 전체를 한 번에 넣지 않고, 검색 가능한 단위로 나눈다.  
 RecursiveCharacterTextSplitter()을 사용한다.  
이 실습에서는 보고서형 PDF에 적합한 기본값으로 `chunk_size=800`, `chunk_overlap=120`을 사용한다.

- `chunk_size`: 하나의 문서 조각 최대 길이
- `chunk_overlap`: 앞뒤 청크가 겹치는 길이
- 목적: 문맥 손실을 줄이면서 검색 단위를 적절히 유지하는 것

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = text_splitter.split_documents(pages)
if not chunks:
    raise ValueError('PDF에서 검색할 텍스트 청크를 만들지 못했습니다.')

for i, doc in enumerate(chunks):
    doc.metadata['chunk_id'] = i
    doc.metadata['page_label'] = doc.metadata.get('page', 0) + 1
    doc.metadata['source_name'] = PDF_NAME

print('전체 청크 수:', len(chunks))
print('첫 번째 청크 메타데이터:')
pprint(chunks[0].metadata)
print('\n첫 번째 청크 내용:')
print(chunks[0].page_content[:800])

전체 청크 수: 27
첫 번째 청크 메타데이터:
{'author': 'user',
 'chunk_id': 0,
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 청크 내용:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [9]:
# 청크 정보를 DataFrame으로 정리하여 페이지별 청크 수와 샘플을 확인
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,2
2,3,2
3,4,2
4,5,2
5,6,2
6,7,2
7,8,2
8,9,2
9,10,2


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,45,"2026년 5월 29일 제 17 호 진화하는 금융사기, 모두의 경계가 필요한 시점"
1,1,2,797,하나은행 ...
2,2,2,358,l 최근에는 AI로 목소리나 화면을 조작하여 피해자가 비대면으로 진위 여부를 확인할...
3,3,3,776,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
4,4,3,266,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...
5,5,4,788,하나은행 ...
6,6,4,415,피해자가 범죄에 연루되는 것을 인지하지 못하도록 인력을 동원할 수 있다는 난점 ● ...
7,7,5,791,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
8,8,5,424,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | ..."
9,9,6,766,하나은행 ...


## 7. 임베딩 모델과 FAISS 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → FAISS 인덱스 저장
```

FAISS는 벡터 간 유사도 검색을 빠르게 수행하기 위한 로컬 벡터 검색 라이브러리이다.  
이 노트북에서는 LangChain의 `FAISS.from_documents()`를 사용해 문서 청크와 임베딩 모델을 연결한다.

In [10]:
embeddings = OpenAIEmbeddings(
    model=os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small'),
    api_key=API_KEY,
    base_url=BASE_URL,
)

INDEX_DIR = Path('faiss_financial_fraud_pdf_index')
FAISS_INDEX_FILE = INDEX_DIR / 'index.faiss'
FAISS_PKL_FILE = INDEX_DIR / 'index.pkl'

if FAISS_INDEX_FILE.exists() and FAISS_PKL_FILE.exists():
    print('기존 FAISS 인덱스를 로딩합니다.')
    # 이 노트북이 직접 생성한 신뢰할 수 있는 인덱스만 로드한다.
    vectorstore = FAISS.load_local(
        str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
    )
    print('FAISS 인덱스 로딩 완료:', INDEX_DIR.resolve())
else:
    print('청크 문서를 임베딩하여 새 FAISS 벡터스토어를 생성합니다.')
    vectorstore = FAISS.from_documents(chunks, embeddings)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))
    print('FAISS 인덱스 생성 및 저장 완료:', INDEX_DIR.resolve())

print('저장된 문서 청크 수:', vectorstore.index.ntotal)

청크 문서를 임베딩하여 새 FAISS 벡터스토어를 생성합니다.
FAISS 인덱스 생성 및 저장 완료: /Users/sunub/workspace/python-lab/gg1th_rag_agent_ex/faiss_financial_fraud_pdf_index
저장된 문서 청크 수: 27


## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 기본 FAISS 설정에서는 거리 기반 점수로 해석한다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [11]:
question = '금융사기는 어떤 단계로 발생하나요?'
results = vectorstore.similarity_search_with_score(question, k=4)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace('\n', ' '))


[1] score=0.8488 | page=13 | chunk=24
● 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전반에  필요한 데이터를 종합적으로 분석하고 학습하여 사기를 예방할 수 있는 시스템 구축 ● 금융회사가 자체적으로 개발한 금융사기 탐지 프로그램을 타 금융사와 공유하는 등  사회적 문제 해결을 위한 공유 체계를 적극적으로 마련하여 피해를 예방하고자 노력 ■ 금융사기는 개인이 순간적으로 사기를 인지하고 예방하기 어렵다는 측면에서  금융회사와 관련 정부 기관의 예방 노력이 함께 이행되어야 함 ● 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므로   금융사기 예방을 위한 범기관 네트워크 구축이 요구 - 통신사, 가상자산거래소, 은행 등 금융사기가 발생하는 지점과 직접적으로 관련된 기관과의  공유 네트워크를 구축함으로써 민관 협력을 통한 예방 체계를 고도화 ● 금융사는 금융사기 전담 부서 조직을 확충하고 금융사기를 예방할 수 있는 시스템을  운영하는 한편, 피해 발생 시 배상 프

[2] score=0.8891 | page=4 | chunk=5
하나은행                                                                                                                    하나금융연구소 33 I. 금융사기란?  1. 금융사기의 정의와 특징 ■ 금융사기는 사람을 속이거나 착각하게 하여 금전적 이득을 취하는 범죄 행위 ● 금융사기는 사기 수법과 수단에 따라 양상이 다르게 나타나나, 공통적으로 ‘신뢰  유도-개인정보 수집-금전 편취’ 단계에 걸쳐 금전적 피해를 유발 - 택배회사 발송 배송조회 QR코드(신뢰)로 위장하여 피해자가 코드 스캔 시 악성코드나  앱을 스마트폰에 설치(수집)하고, 이를 통해 탈취한 개인정보로 금전을 유출(탈취) - 유명 핀플루언서(금융+인플루언서)의 영상을 도용(신뢰)하여 가짜 채널로 가

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

retrieved_docs = retriever.invoke('전기통신금융사기와 투자사기의 차이는 무엇인가요?')

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace('\n', ' '))
    print('-' * 80)

page=7, chunk=12
- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안심차단(’25.11)을  차례로 시행하여 금융소비자가 자발적 신청을 통해 금융사기를 예방하도록 장려 ● 금융소비자가 직접 신청해야 하는 서비스 외에 자금 이체 시 수취인 계좌에 일정 시간  경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입 표3 | 전기통신금융사기 주요 대응 현황 표4 | 투자사기 주요 대응 현황 시기 주요 내용 시기 주요 내용 ’23.7 통합신고대응센터 개소 ’24.1 자본시장법  개정  Ÿ 불공정거래 
--------------------------------------------------------------------------------
page=5, chunk=7
Hana Bank Hana Institute of Finance 4 2. 금융사기 주요 유형 ■ 금융사기는 금전적 피해를 유발하는 범죄행위 전반을 포괄해 다양한 기준으로  구분되나 전기통신금융사기와 투자사기 두 분류로 구분하는 것이 보편적 ● 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용하여 금융기관·    지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법 - 피싱, 스미싱, 파밍 등이 대표적인 유형이며, 법에서 열거하는 통신수단을 이용하여   개인정보를 노출시켜 자금을 송금·이체하도록 유도 - ’
--------------------------------------------------------------------------------
page=5, chunk=8
사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | 금융사기 분류 그림2 | 금융사기 접근 경로 분류 하위 종류 전기통신 금융사기 보이스피싱 전화로 타인을 사칭하여 송금을 유도하는 수법 메신저피싱 카카오톡이나 SNS로 타인을 사칭하여 송금을  유도하는 수법 파밍 가짜 사이트에 접속하여 개인정보를 입력하도

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [13]:
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get('page_label', '?')
        chunk_id = doc.metadata.get('chunk_id', '?')
        formatted.append(f'[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}')
    return '\n\n'.join(formatted)


prompt = ChatPromptTemplate.from_messages([
    (
        'system',
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip(),
    ),
    ('human', '{question}'),
])

llm = ChatOpenAI(
    model=os.getenv('CHAT_MODEL', 'gpt-4.1' if BASE_URL else 'gpt-4o-mini'),
    temperature=0,
    api_key=API_KEY,
    base_url=BASE_URL,
    use_responses_api=False,
)
answer_chain = prompt | llm | StrOutputParser()

In [14]:
question = '전기통신금융사기와 투자사기는 어떻게 다른가요?'
docs = vectorstore.similarity_search(question, k=4)
context = format_docs(docs)
answer = answer_chain.invoke({'question': question, 'context': context})

OpenAIInvalidRequestError: Error code: 400 - {'error': {'type': 'invalid_request_error', 'message': 'MonoRouter Universal: the requested model is not supported. Supported models: gpt-5-nano-chat, gpt-6-astra, gpt-6.1-sol, gpt-6-sol, gpt-6-luna, gpt-5.6-sol, gpt-5.6-terra, gpt-5.6-luna, gpt-5.5, gpt-5.4, gpt-5.4-mini, gpt-5.4-mini-chat, gpt-5.4-nano, gpt-5.4-nano-chat, gpt-4.1, gpt-4o, claude-haiku-4.5, claude-fable-5-1, claude-fable-5, claude-sonnet-5, claude-opus-5-5, claude-opus-5, claude-opus-4-8, claude-opus-4-7, claude-opus-4-6, claude-sonnet-4-6, gemini-3.8-flash, gemini-3.8-flash-google-search, gemini-3.7-flash, gemini-3.7-flash-google-search, gemini-3.6-flash, gemini-3.6-flash-search, gemini-3.5-flash-lite, gemini-3.5-flash-lite-search, gemini-3.5-flash, gemini-3.5-flash-search, gemini-2.5-flash-lite, gemini-3.1-pro-preview.', 'user_message': '지원하지 않는 모델입니다. 사용 가능 모델: gpt-5-nano-chat, gpt-6-astra, gpt-6.1-sol, gpt-6-sol, gpt-6-luna, gpt-5.6-sol, gpt-5.6-terra, gpt-5.6-luna, gpt

In [ ]:
# 결과 출력
answer

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ FAISS 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [ ]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 FAISS 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [ ]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)

print("\n[검색된 근거]")
display(sources)  # Jupyter 환경에서는 display로 테이블이 예쁘게 나옵니다.


In [ ]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [ ]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

## 14. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩
→ 페이지 단위 Document 생성
→ RecursiveCharacterTextSplitter로 청킹
→ OpenAIEmbeddings로 임베딩
→ FAISS 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | FAISS 저장·로드 구조로 인덱싱 비용 절감 가능 |